<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-06-function-calling/lesson-6.3-parallel-tools/notebooks/GCP_Capstone_6.3_ParallelTools.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 6.3 Parallel Calls & Built-in Tools — Google Search + Code Exec + Custom
**Netsetos GenAI Engineering — GCP Capstone**

Execute parallel calls concurrently. Combine Google Search, Code Execution, and custom functions.


## Setup


In [ ]:
!pip install -q google-genai

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project=PROJECT_ID,
                      location='us-central1')

# Reuse DocuMind functions from Lesson 6.1-6.2
def search_documents(query: str, doc_type: str = 'all') -> dict:
    """Search DocuMind internal documents."""
    mock = {'legal': [{'id': 'D-01', 'title': 'Privacy Policy v3', 'pages': 12}]}
    results = mock.get(doc_type, [{'id': 'D-99', 'title': f'Result for: {query}', 'pages': 5}])
    return {'documents': results, 'total': len(results)}

def calculate_processing_cost(num_documents: int, total_pages: int, processing_type: str = 'standard') -> dict:
    """Estimate document processing cost."""
    rates = {'standard': 0.05, 'priority': 0.12, 'bulk': 0.03}
    cost = total_pages * rates.get(processing_type, 0.05)
    return {'cost_usd': round(cost, 2), 'cost_inr': round(cost * 84, 2)}

def get_usage_stats(metric: str, days: int = 7) -> dict:
    """Get RAG pipeline usage statistics."""
    return {'metric': metric, 'period': f'last {days} days', 'value': 1247}

TOOLS = [search_documents, calculate_processing_cost, get_usage_stats]
print('Ready')


## Cell 1: Google Search Grounding


In [ ]:
# Google Search: model queries the web automatically
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='What are the latest data protection regulations in India for 2026?',
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())])
)
print('=== Google Search Grounded Response ===')
print(response.text[:300])

# Check grounding metadata
metadata = response.candidates[0].grounding_metadata
if metadata:
    print(f'\nSearch queries: {metadata.web_search_queries}')
    if metadata.grounding_chunks:
        for chunk in metadata.grounding_chunks[:3]:
            print(f'  Source: {chunk.web.title}')
            print(f'  URL: {chunk.web.uri}')


## Cell 2: Code Execution


In [ ]:
# Code Execution: Gemini generates + runs Python
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Calculate compound interest on Rs 10,00,000 at 8.5% '
             'for 5 years compounded quarterly. Show the formula and result.',
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.ToolCodeExecution)])
)

print('=== Code Execution Response ===')
for part in response.candidates[0].content.parts:
    if part.executable_code:
        print(f'Generated code:\n{part.executable_code.code}\n')
    if part.code_execution_result:
        print(f'Execution result: {part.code_execution_result.output}')
    if part.text:
        print(f'Explanation: {part.text[:200]}')


## Cell 3: Parallel Function Calls


In [ ]:
# Trigger parallel calls with an independent multi-part query
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Search for our legal documents AND show this week query stats',
    config=types.GenerateContentConfig(
        tools=TOOLS,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
)

print('=== Parallel Calls ===')
if response.function_calls:
    print(f'Number of calls: {len(response.function_calls)}')
    for fc in response.function_calls:
        print(f'  {fc.name}({dict(fc.args)}) id={fc.id}')
else:
    print('No function calls (model responded with text)')


## Cell 4: Concurrent Execution


In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def execute_parallel_sync(function_calls, functions):
    """Execute multiple function calls concurrently."""
    start = time.time()
    results = [None] * len(function_calls)
    with ThreadPoolExecutor(max_workers=len(function_calls)) as pool:
        futures = {
            pool.submit(functions[fc.name], **fc.args): i
            for i, fc in enumerate(function_calls)
        }
        for future in futures:
            i = futures[future]
            fc = function_calls[i]
            try:
                result = future.result(timeout=15)
                results[i] = types.Part.from_function_response(
                    name=fc.name, response={'result': result}, id=fc.id)
            except Exception as e:
                results[i] = types.Part.from_function_response(
                    name=fc.name, response={'error': str(e)}, id=fc.id)
    elapsed = time.time() - start
    print(f'Parallel execution: {elapsed:.2f}s for {len(function_calls)} calls')
    return results

# Test with the parallel calls from Cell 3
if response.function_calls and len(response.function_calls) > 1:
    FUNCTIONS = {'search_documents': search_documents,
                 'calculate_processing_cost': calculate_processing_cost,
                 'get_usage_stats': get_usage_stats}
    result_parts = execute_parallel_sync(
        list(response.function_calls), FUNCTIONS)
    print(f'Got {len(result_parts)} results')
    for rp in result_parts:
        print(f'  {rp.function_response.name}: {rp.function_response.response}')


## Cell 5: Google Search + Custom Functions Combined


In [ ]:
# Combine Google Search with custom document search
chat = client.chats.create(
    model='gemini-2.5-flash',
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(google_search=types.GoogleSearch()),
            types.Tool(function_declarations=[
                types.FunctionDeclaration(
                    name='search_documents',
                    description='Search internal company documents by keyword.',
                    parameters={'type': 'object',
                                'properties': {'query': {'type': 'string'},
                                               'doc_type': {'type': 'string', 'enum': ['legal', 'invoice', 'all']}},
                                'required': ['query']})
            ]),
        ],
        system_instruction='You are DocuMind AI. Use search_documents for '
                           'internal company documents. Use Google Search for '
                           'external regulations and laws.'
    )
)

# Test: internal question
r1 = chat.send_message('Find our privacy policy')
print(f'Internal query: {r1.text[:150]}...' if len(r1.text) > 150 else f'Internal: {r1.text}')

# Test: external question  
r2 = chat.send_message('What are the latest GDPR updates?')
print(f'\nExternal query: {r2.text[:150]}...' if len(r2.text) > 150 else f'\nExternal: {r2.text}')


## Cell 6: Code Execution + Custom Functions


In [ ]:
# Custom function for data + Code Execution for computation
chat2 = client.chats.create(
    model='gemini-2.5-flash',
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(code_execution=types.ToolCodeExecution),
            *TOOLS,  # Custom functions
        ],
        system_instruction='You are DocuMind AI. Use custom functions to get '
                           'document data. Use Code Execution for calculations '
                           'and statistical analysis on that data.'
    )
)

r = chat2.send_message('How much would it cost to process 500 pages at '
                       'standard rate? Calculate the monthly cost if we '
                       'process this amount weekly.')
print(f'Response: {r.text[:300]}')


## Cell 7: Multi-Tool DocuMind Agent


In [ ]:
# Complete multi-tool agent
SYSTEM_PROMPT = '''You are DocuMind AI, a document intelligence assistant.
TOOL ROUTING:
- search_documents: internal company documents
- calculate_processing_cost: cost estimation
- get_usage_stats: pipeline analytics
- Google Search: external regulations, industry benchmarks
- Code Execution: calculations, statistics, charts
Always use the most appropriate tool for each query part.'''

agent_config = types.GenerateContentConfig(
    tools=[
        types.Tool(google_search=types.GoogleSearch()),
        types.Tool(code_execution=types.ToolCodeExecution),
        *TOOLS,
    ],
    system_instruction=SYSTEM_PROMPT
)

# Test different routing scenarios
test_queries = [
    'Find our legal documents',                    # Custom only
    'What is the current DPDP Act in India?',      # Google Search
    'Calculate 15% tax on Rs 2,50,000',            # Code Execution
    'Hello, how can you help me?',                  # No tools (text)
]

for q in test_queries:
    r = client.models.generate_content(
        model='gemini-2.5-flash', contents=q, config=agent_config)
    tool_used = 'function_call' if r.function_calls else 'text/built-in'
    print(f'Q: {q}')
    print(f'  Tool: {tool_used}')
    print(f'  A: {r.text[:100] if r.text else "[function call pending]"}\n')


## ✅ Lesson 6.3 Complete! MODULE 6 COMPLETE!

**Parallel execution mastered:**
- ✅ asyncio.gather and ThreadPoolExecutor patterns
- ✅ ID-based result correlation
- ✅ Partial failure handling (1:1 parity)

**Built-in tools mastered:**
- ✅ Google Search grounding with metadata + citations
- ✅ Code Execution sandbox (numpy, pandas, matplotlib)
- ✅ Multi-tool combination in single request
- ✅ System instruction routing for tool selection

**Module 6 Complete — 3 Lessons:**
- 6.1: FunctionDeclarations, AUTO/ANY/NONE modes
- 6.2: While-loop dispatcher, sequential chaining, FunctionRegistry
- 6.3: Parallel execution, Google Search + Code Exec + custom tools

**Next: Module 7 — MCP Servers on Cloud Run**
